In [ ]:

# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES
# TO THE CORRECT LOCATION (/kaggle/input) IN YOUR NOTEBOOK,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

import os
import sys
from tempfile import NamedTemporaryFile
from urllib.request import urlopen
from urllib.parse import unquote, urlparse
from urllib.error import HTTPError
from zipfile import ZipFile
import tarfile
import shutil

CHUNK_SIZE = 40960
DATA_SOURCE_MAPPING = 't20-wc:https%3A%2F%2Fstorage.googleapis.com%2Fkaggle-data-sets%2F5407732%2F8980530%2Fbundle%2Farchive.zip%3FX-Goog-Algorithm%3DGOOG4-RSA-SHA256%26X-Goog-Credential%3Dgcp-kaggle-com%2540kaggle-161607.iam.gserviceaccount.com%252F20240718%252Fauto%252Fstorage%252Fgoog4_request%26X-Goog-Date%3D20240718T094540Z%26X-Goog-Expires%3D259200%26X-Goog-SignedHeaders%3Dhost%26X-Goog-Signature%3Da2751c00cee4d4ef3a09f2dd06ecdeb2bcae88dbfcea737923c78fec6eeb133b5f9eb3d30573f4d7904e65f32dd8335757c87391e852bca9b3734f92c5f1ed6fa2ff772eee2bb59e51f5d44cc34e0b639f7f0d062dcc781f5a4e2841713126d64628c4c830e0001fdcb15adc466b05dd92ce9869237195ea5281bcd3a7189de77ac871d82f05482c2257f7b2acfaa0da36415d84fd965925c79a2f7b4444d89ba85d9f94c93794c0a923cfee3efb1300efca104cfe7d26f0480342d118c31ba28f1de9e84026a808746ccb3ea613172635810a37efc31081f05ca57642aec2b13d9e7b3b9d2b6c18846392cb044cdcc2aafcde86f63f8f66ebae6e8102ef5615,ougcgutxyreztew:https%3A%2F%2Fstorage.googleapis.com%2Fkaggle-data-sets%2F5409386%2F8982673%2Fbundle%2Farchive.zip%3FX-Goog-Algorithm%3DGOOG4-RSA-SHA256%26X-Goog-Credential%3Dgcp-kaggle-com%2540kaggle-161607.iam.gserviceaccount.com%252F20240718%252Fauto%252Fstorage%252Fgoog4_request%26X-Goog-Date%3D20240718T094540Z%26X-Goog-Expires%3D259200%26X-Goog-SignedHeaders%3Dhost%26X-Goog-Signature%3D14f4942959f70782054eb44d6a5c9e0f77f2d38ef1f33455e7a9ad432be3069c3be4204ad610ca821a748536d4730da1e5c348e7fc2c8039570d7e6bf54d677e21aa75982884ef7dc59c7513d389c0fb254e06ba7b58ed4bdd438c99065eae1242ca6b763e84d1a8a38d34e2faaf2f63d7ea661d537afa96b9134fd38b36cf408d14bd884cc7b524cc04deb82cfba98e55b67cd159fda5f4d0792aabf11385c9ef70651b9efc7d5fdf79a8a1d37681871c140081287d4df5f089566eba975650b7260ba8a69a0bfb9f7f7a4ed42876be94f2e08ed14095b3baca927c26ceedb5e7b5fc638c2bb61ab2d4e02ce9521244207c42a660f9ea976a2350c2c9d8412f'

KAGGLE_INPUT_PATH='/kaggle/input'
KAGGLE_WORKING_PATH='/kaggle/working'
KAGGLE_SYMLINK='kaggle'

!umount /kaggle/input/ 2> /dev/null
shutil.rmtree('/kaggle/input', ignore_errors=True)
os.makedirs(KAGGLE_INPUT_PATH, 0o777, exist_ok=True)
os.makedirs(KAGGLE_WORKING_PATH, 0o777, exist_ok=True)

try:
  os.symlink(KAGGLE_INPUT_PATH, os.path.join("..", 'input'), target_is_directory=True)
except FileExistsError:
  pass
try:
  os.symlink(KAGGLE_WORKING_PATH, os.path.join("..", 'working'), target_is_directory=True)
except FileExistsError:
  pass

for data_source_mapping in DATA_SOURCE_MAPPING.split(','):
    directory, download_url_encoded = data_source_mapping.split(':')
    download_url = unquote(download_url_encoded)
    filename = urlparse(download_url).path
    destination_path = os.path.join(KAGGLE_INPUT_PATH, directory)
    try:
        with urlopen(download_url) as fileres, NamedTemporaryFile() as tfile:
            total_length = fileres.headers['content-length']
            print(f'Downloading {directory}, {total_length} bytes compressed')
            dl = 0
            data = fileres.read(CHUNK_SIZE)
            while len(data) > 0:
                dl += len(data)
                tfile.write(data)
                done = int(50 * dl / int(total_length))
                sys.stdout.write(f"\r[{'=' * done}{' ' * (50-done)}] {dl} bytes downloaded")
                sys.stdout.flush()
                data = fileres.read(CHUNK_SIZE)
            if filename.endswith('.zip'):
              with ZipFile(tfile) as zfile:
                zfile.extractall(destination_path)
            else:
              with tarfile.open(tfile.name) as tarfile:
                tarfile.extractall(destination_path)
            print(f'\nDownloaded and uncompressed: {directory}')
    except HTTPError as e:
        print(f'Failed to load (likely expired) {download_url} to path {destination_path}')
        continue
    except OSError as e:
        print(f'Failed to load {download_url} to path {destination_path}')
        continue

print('Data source import complete.')


## RAG System Using Llama2 With Hugging Face

In [ ]:
!pip install pypdf

In [ ]:
!pip install -q transformers einops accelerate langchain bitsandbytes

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf 24.6.1 requires cubinlinker, which is not installed.
cudf 24.6.1 requires cupy-cuda11x>=12.0.0, which is not installed.
cudf 24.6.1 requires ptxcompiler, which is not installed.
cuml 24.6.1 requires cupy-cuda11x>=12.0.0, which is not installed.
dask-cudf 24.6.1 requires cupy-cuda11x>=12.0.0, which is not installed.
keras-cv 0.9.0 requires keras-core, which is not installed.
tensorflow-decision-forests 1.8.1 requires wurlitzer, which is not installed.
apache-beam 2.46.0 requires dill<0.3.2,>=0.3.1.1, but you have dill 0.3.8 which is incompatible.
apache-beam 2.46.0 requires numpy<1.25.0,>=1.14.3, but you have numpy 1.26.4 which is incompatible.
apache-beam 2.46.0 requires pyarrow<10.0.0,>=3.0.0, but you have pyarrow 16.1.0 which is incompatible.
cudf 24.6.1 requires cuda-python<12.0a0,>=11.7.1, but you have cu

In [ ]:
## Embedding
!pip install -q sentence_transformers

In [ ]:
!pip install -q llama-index-llms-huggingface

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
preprocessing 0.1.13 requires nltk==3.2.4, but you have nltk 3.8.1 which is incompatible.


In [ ]:
!pip install -q llama-index-readers-file # Install the missing package for file readers

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
libpysal 4.9.2 requires shapely>=2.0.1, but you have shapely 1.8.5.post1 which is incompatible.
momepy 0.7.2 requires shapely>=2, but you have shapely 1.8.5.post1 which is incompatible.
pointpats 2.5.0 requires shapely>=2, but you have shapely 1.8.5.post1 which is incompatible.
spaghetti 1.7.6 requires shapely>=2.0.1, but you have shapely 1.8.5.post1 which is incompatible.
spopt 0.6.1 requires shapely>=2.0.1, but you have shapely 1.8.5.post1 which is incompatible.


In [ ]:
from llama_index.core import VectorStoreIndex,SimpleDirectoryReader,ServiceContext,PromptTemplate
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.core.prompts.prompts import SimpleInputPrompt

/opt/conda/lib/python3.10/site-packages/pydantic/_internal/_fields.py:161: UserWarning: Field "model_id" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [ ]:
documents=SimpleDirectoryReader("/kaggle/input/ougcgutxyreztew").load_data()

In [ ]:
system_prompt="""
You are an expert assistant. Answer the question based on the given context.
DON'T REFER TO THE PREVIOUS ANSWERS FOR THE CURRENT ANSWERS.
"""
## Default format supportable by LLama2
query_wrapper_prompt=SimpleInputPrompt("<|USER|>{query_str}<|ASSISTANT|>")

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

# # option 2: key login
# from huggingface_hub import login
# write_key = 'hf_' # paste token here
# login(write_key)

In [ ]:
import torch

llm = HuggingFaceLLM(
    context_window=4096,
    max_new_tokens=2048,
    generate_kwargs={"temperature": 0.0, "do_sample": False},
    system_prompt=system_prompt,
    query_wrapper_prompt=query_wrapper_prompt,
    tokenizer_name="google/gemma-2-9b",
    model_name="google/gemma-2-9b",
    device_map="auto",
    # uncomment this if using CUDA to reduce memory usage
    model_kwargs={"torch_dtype": torch.float16 , "load_in_8bit":True}
)

config.json:   0%|          | 0.00/856 [00:00<?, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


model.safetensors.index.json:   0%|          | 0.00/39.1k [00:00<?, ?B/s]

model-00001-of-00008.safetensors:   0%|          | 0.00/4.84G [00:00<?, ?B/s]

model-00002-of-00008.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00003-of-00008.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

... (7 outputs left out)

tokenizer_config.json:   0%|          | 0.00/40.0k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

In [ ]:
!pip install -q langchain-community langchain-core

In [ ]:
!pip install -q -U llama-index-core llama-index-llms-openai llama-index-embeddings-openai

In [ ]:
!pip install -q llama-index-legacy

In [ ]:
from llama_index.core import ServiceContext

In [ ]:
from langchain.embeddings.huggingface import HuggingFaceEmbeddings
# from llama_index import ServiceContext
from llama_index.legacy.embeddings.langchain import LangchainEmbedding

embed_model=LangchainEmbedding(
    HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2"))

[nltk_data] Downloading package stopwords to
[nltk_data]     /opt/conda/lib/python3.10/site-
[nltk_data]     packages/llama_index/legacy/_static/nltk_cache...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     /opt/conda/lib/python3.10/site-
[nltk_data]     packages/llama_index/legacy/_static/nltk_cache...
[nltk_data]   Unzipping tokenizers/punkt.zip.
/opt/conda/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  warn_deprecated(
2024-07-18 09:43:45.636308: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting 

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

... (2 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
service_context=ServiceContext.from_defaults(
    chunk_size=1024,
    llm=llm,
    embed_model=embed_model
)

/tmp/ipykernel_34/2250285859.py:1: DeprecationWarning: Call to deprecated class method from_defaults. (ServiceContext is deprecated, please use `llama_index.settings.Settings` instead.) -- Deprecated since version 0.10.0.
  service_context=ServiceContext.from_defaults(


In [ ]:
service_context

ServiceContext(llm_predictor=LLMPredictor(system_prompt=None, query_wrapper_prompt=None, pydantic_program_mode=<PydanticProgramMode.DEFAULT: 'default'>), prompt_helper=PromptHelper(context_window=4096, num_output=2048, chunk_overlap_ratio=0.1, chunk_size_limit=None, separator=' '), embed_model=LangchainEmbedding(model_name='sentence-transformers/all-mpnet-base-v2', embed_batch_size=10, callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x7c567cd1af20>), transformations=[SentenceSplitter(include_metadata=True, include_prev_next_rel=True, callback_manager=<llama_index.core.callbacks.base.CallbackManager object at 0x7c567cd1af20>, id_func=<function default_id_func at 0x7c57c0bfc8b0>, chunk_size=1024, chunk_overlap=200, separator=' ', paragraph_separator='\n\n\n', secondary_chunking_regex='[^,.;。？！]+[,.;。？！]?')], llama_logger=<llama_index.core.service_context_elements.llama_logger.LlamaLogger object at 0x7c56f04d3a60>, callback_manager=<llama_index.core.callbacks.b

In [ ]:
index=VectorStoreIndex.from_documents(documents,service_context=service_context)

In [ ]:
index

In [ ]:
query_engine=index.as_query_engine()

In [ ]:
response=query_engine.query("Who won the match between Sri Lanka vs West Indies on 2016-03-20?")

/opt/conda/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:540: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


In [ ]:
type(response)

llama_index.core.base.response.schema.Response

In [ ]:
response = query_engine.query("Who won the match between Sri Lanka vs West Indies on 2016-03-20?")

# Extract and print the relevant part of the response
if response and hasattr(response, 'source_nodes'):
    for node_with_score in response.source_nodes:
        if hasattr(node_with_score, 'node') and hasattr(node_with_score.node, 'text'):
            # Extract the text content
            text = node_with_score.node.text
            # Find the result line in the text
            for line in text.split('\n'):
                if line.startswith('Result:'):
                    print(line)
                    break
else:
    print("No relevant response found.")


Result: West Indies won by 7 wickets
Result: Sri Lanka won by 27 runs


In [ ]:
response = query_engine.query("Who won the match between Namibia vs India on 2021-11-08?")

# Extract and print the relevant part of the response
if response and hasattr(response, 'source_nodes'):
    for node_with_score in response.source_nodes:
        if hasattr(node_with_score, 'node') and hasattr(node_with_score.node, 'text'):
            # Extract the text content
            text = node_with_score.node.text
            # Find the result line in the text
            for line in text.split('\n'):
                if line.startswith('Result:'):
                    print(line)
                    break
else:
    print("No relevant response found.")


Result: India won by 9 wickets
Result: Sri Lanka won by 7 wickets


In [ ]:
response = query_engine.query("Who was the man of the match between New Zealand vs Namibia happened in 2021?")

# Extract and print the relevant part of the response
if response and hasattr(response, 'source_nodes'):
    for node_with_score in response.source_nodes:
        if hasattr(node_with_score, 'node') and hasattr(node_with_score.node, 'text'):
            # Extract the text content
            text = node_with_score.node.text
            # Find and print the "Man of the Match" line in the text
            for line in text.split('\n'):
                if '0.5' in line:
                    print(line)
                    break
else:
    print("No relevant response found.")


0.5: MJ Guptill faced BM Scholtz, scored 0 runs. Team score: 8/0


In [ ]:
response = query_engine.query("Who won the match between India vs Bangladesh happened in 2016?")

# Extract and print the relevant part of the response
if response and hasattr(response, 'source_nodes'):
    for node_with_score in response.source_nodes:
        if hasattr(node_with_score, 'node') and hasattr(node_with_score.node, 'text'):
            # Extract the text content
            text = node_with_score.node.text
            # Print the entire text for debugging purposes
            print("Full text content:\n", text)

            # Find the result line in the text
            for line in text.split('\n'):
                # Print each line for debugging purposes
                print("Checking line:", line)
                if line.startswith('9.4:'):
                    print("Match result:", line)
                    break
else:
    print("No relevant response found.")


Full text content:
 Match: World T20 - T20
Date: 2016-03-23
Venue: M Chinnaswamy Stadium, Bangalore
Teams: India vs Bangladesh
Umpires: Aleem Dar, BNJ Oxenford
Toss: Bangladesh won the toss and decided to field
Result: India won by 1 runs
Man of the Match: R Ashwin

Innings 1:
0.1: RG Sharma faced Mashrafe Mortaza, scored 1 runs. Team score: 1/0
0.2: S Dhawan faced Mashrafe Mortaza, scored 1 runs. Team score: 2/0
0.3: RG Sharma faced Mashrafe Mortaza, scored 1 runs. Team score: 3/0
0.4: RG Sharma faced Mashrafe Mortaza, scored 0 runs. Team score: 3/0
0.5: RG Sharma faced Mashrafe Mortaza, scored 1 runs. Team score: 4/0
0.6: S Dhawan faced Mashrafe Mortaza, scored 1 runs. Team score: 5/0
0.7: RG Sharma faced Mashrafe Mortaza, scored 0 runs. Team score: 5/0
1.1: S Dhawan faced Shuvagata Hom, scored 0 runs. Team score: 5/0
1.2: S Dhawan faced Shuvagata Hom, scored 0 runs. Team score: 5/0
1.3: S Dhawan faced Shuvagata Hom, scored 0 runs. Team score: 5/0
1.4: S Dhawan faced Shuvagata Hom, s